In [10]:
import requests
from bs4 import BeautifulSoup
import re
from typing import List, Dict, Optional
import time
from urllib.parse import urljoin
import json
import pandas as pd

class KDNuggetsScraper:
    def __init__(self, base_url: str = "https://www.kdnuggets.com"):
        self.base_url = base_url
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        })
    
    def fetch_page(self, url: str) -> Optional[BeautifulSoup]:
        """Fetch and parse a webpage"""
        try:
            response = self.session.get(url, timeout=10)
            response.raise_for_status()
            return BeautifulSoup(response.content, 'html.parser')
        except requests.RequestException as e:
            print(f"Error fetching {url}: {e}")
            return None
    
    def extract_article_info_from_listing(self, soup: BeautifulSoup) -> List[Dict]:
        """Extract article information from the news listing page"""
        articles = []
        
        # Find articles using the table.thb structure
        rows = soup.select('table.thb ul li.li-has-thumb')
        for row in rows:
            article = {
                'url': row.find('a')['href'] if row.find('a') else '',
                'title': row.find('a').get_text().strip() if row.find('a') else '',
                'description': row.find('font').get_text().strip() if row.find('font') else '',
                'author': row.find('div', class_='author-link').find('a').get_text().strip() if row.find('div', class_='author-link') else '',
                'date': '',
                'tags': []
            }
            
            # Make URL absolute if it's relative
            if article['url'] and article['url'].startswith('/'):
                article['url'] = urljoin(self.base_url, article['url'])
                
            if article['url'] and article['title']:
                articles.append(article)
        
        # If no articles found with primary method, fall back to other methods
        if not articles:
            # Look for links to articles in the main content area
            content_area = soup.find(['main', 'div'], class_=re.compile(r'.*(?:content|main|posts).*', re.I))
            if content_area:
                article_links = content_area.find_all('a', href=re.compile(r'/[^/]+/?$'))
                for link in article_links:
                    if self._is_valid_article_link(link):
                        article_info = self._extract_info_from_link(link)
                        if article_info:
                            articles.append(article_info)
        
        # If still no articles found, try parsing text content
        if not articles:
            text_content = soup.get_text()
            articles.extend(self._parse_text_content(text_content))
        
        return articles
    
    def _parse_text_content(self, text: str) -> List[Dict]:
        """Parse article information from raw text content"""
        articles = []
        
        # Split by lines and look for article patterns
        lines = text.split('\n')
        current_article = {}
        
        for i, line in enumerate(lines):
            line = line.strip()
            if not line:
                continue
            
            # Check if line contains a URL pattern (article link)
            url_match = re.search(r'https://www\.kdnuggets\.com/([^)]+)', line)
            if url_match:
                # If we have a current article, save it
                if current_article:
                    articles.append(current_article)
                
                # Start new article
                url = url_match.group(0).rstrip(')')
                title = self._extract_title_from_line(line)
                
                current_article = {
                    'url': url,
                    'title': title,
                    'description': '',
                    'author': '',
                    'date': '',
                    'tags': []
                }
                
                # Look for description in the next line
                if i + 1 < len(lines):
                    next_line = lines[i + 1].strip()
                    if next_line and not next_line.startswith('By ') and not next_line.startswith('http'):
                        current_article['description'] = next_line
            
            # Check for author information
            elif line.startswith('By ') and current_article:
                author_info = self._extract_author_info(line)
                current_article.update(author_info)
        
        # Add the last article if exists
        if current_article:
            articles.append(current_article)
        
        return articles
    
    def _extract_title_from_line(self, line: str) -> str:
        """Extract title from a line containing URL"""
        # Remove URL and clean up
        title = re.sub(r'https://www\.kdnuggets\.com/[^\s\]]+', '', line)
        # Remove markdown link syntax
        title = re.sub(r'[\[\]()]', '', title)
        # Clean up extra spaces and dashes
        title = re.sub(r'^-+\s*', '', title)
        title = re.sub(r'\s*-+$', '', title)
        # Replace multiple whitespaces (including newlines) with a single space
        title = re.sub(r'\s+', ' ', title)
        return title.strip()
    
    def _extract_author_info(self, line: str) -> Dict:
        """Extract author, date, and tags from author line"""
        info = {'author': '', 'date': '', 'tags': []}
        
        # Pattern: By [Author Name](url), Title on Date in [Tags]
        author_match = re.search(r'By \[([^\]]+)\]', line)
        if author_match:
            info['author'] = author_match.group(1)
        
        # Extract date
        date_match = re.search(r'on ([A-Za-z]+ \d{1,2}, \d{4})', line)
        if date_match:
            info['date'] = date_match.group(1)
        
        # Extract tags
        tags_match = re.search(r'in \[([^\]]+)\]', line)
        if tags_match:
            info['tags'] = [tag.strip() for tag in tags_match.group(1).split(',')]
        
        return info
    
    def _is_valid_article_link(self, link) -> bool:
        """Check if link is a valid article link"""
        href = link.get('href', '')
        text = link.get_text().strip()
        
        return (
            href and 
            'kdnuggets.com' in href and 
            text and 
            len(text) > 10 and
            not any(skip in href.lower() for skip in ['author', 'tag', 'category', 'about'])
        )
    
    def _extract_info_from_link(self, link) -> Optional[Dict]:
        """Extract information from a single article link"""
        href = link.get('href', '')
        title = link.get_text().strip()
        
        if not href or not title:
            return None
        
        # Make URL absolute
        if href.startswith('/'):
            href = urljoin(self.base_url, href)
        
        return {
            'url': href,
            'title': title,
            'description': '',
            'author': '',
            'date': '',
            'tags': []
        }
    
    def get_full_article_details(self, article_url: str) -> Dict:
        """Fetch full article details from individual article page"""
        soup = self.fetch_page(article_url)
        if not soup:
            return {}
        
        details = {}
        
        # Extract title
        title = soup.find('h1')
        if title:
            details['title'] = title.get_text().strip()
        
        # Extract description/summary
        meta_desc = soup.find('meta', attrs={'name': 'description'})
        if meta_desc:
            details['description'] = meta_desc.get('content', '')
        else:
            # Try to find first paragraph
            first_p = soup.find('p')
            if first_p:
                details['description'] = first_p.get_text().strip()[:200] + '...'
        
        # Extract author
        author_elem = soup.find(['span', 'div', 'a'], class_=re.compile(r'.*author.*', re.I))
        if author_elem:
            details['author'] = author_elem.get_text().strip()
        
        # Extract date
        date_elem = soup.find(['time', 'span', 'div'], class_=re.compile(r'.*(?:date|time|published).*', re.I))
        if date_elem:
            details['date'] = date_elem.get_text().strip()
        
        # Extract tags
        tag_elements = soup.find_all(['a', 'span'], class_=re.compile(r'.*(?:tag|category).*', re.I))
        details['tags'] = [tag.get_text().strip() for tag in tag_elements if tag.get_text().strip()]
        
        return details
    
    def scrape_news_page(self, url: str = "https://www.kdnuggets.com/news/index.html", 
                        get_full_details: bool = False, max_articles: int = None) -> List[Dict]:
        """
        Main method to scrape the KDNuggets news page
        
        Args:
            url: URL to scrape
            get_full_details: Whether to fetch full details for each article
            max_articles: Maximum number of articles to process
        
        Returns:
            List of article dictionaries
        """
        print(f"Scraping {url}...")
        soup = self.fetch_page(url)
        
        if not soup:
            return []
        
        articles = self.extract_article_info_from_listing(soup)
        
        if max_articles:
            articles = articles[:max_articles]
        
        print(f"Found {len(articles)} articles")
        
        # Get full details if requested
        if get_full_details:
            print("Fetching full article details...")
            for i, article in enumerate(articles):
                print(f"Processing article {i+1}/{len(articles)}: {article.get('title', 'Unknown')[:50]}...")
                
                full_details = self.get_full_article_details(article['url'])
                article.update(full_details)
                
                # Be respectful with requests
                time.sleep(1)
        
        return articles
    
    def save_to_json(self, articles: List[Dict], filename: str = '../datasets/kdnuggets_articles.json'):
        """Save articles to JSON file"""
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(articles, f, indent=2, ensure_ascii=False)
        print(f"Saved {len(articles)} articles to {filename}")
    
    def to_dataframe(self, articles: List[Dict]) -> 'pd.DataFrame':
        """Convert articles to pandas DataFrame"""
        # Create DataFrame with all available fields and proper column names
        df = pd.DataFrame(articles)
        
        # Rename columns to match the expected format
        column_mapping = {
            'title': 'Title',
            'description': 'Description',
            'author': 'Author',
            'date': 'Date',
            'url': 'URL',
        }
        
        df = df.rename(columns=column_mapping)
        
        # Convert tags list to string if present
        if 'tags' in df.columns:
            df['Tags'] = df['tags'].apply(lambda x: ', '.join(x) if isinstance(x, list) else '')
            df = df.drop('tags', axis=1)
        
        return df
    
    def print_summary(self, articles: List[Dict]):
        """Print a summary of scraped articles"""
        print(f"\n=== SCRAPED {len(articles)} ARTICLES ===\n")
        
        for i, article in enumerate(articles, 1):
            print(f"{i}. {article.get('title', 'No title')}")
            print(f"   URL: {article.get('url', 'No URL')}")
            if article.get('description'):
                print(f"   Description: {article['description'][:100]}...")
            if article.get('author'):
                print(f"   Author: {article['author']}")
            if article.get('date'):
                print(f"   Date: {article['date']}")
            if article.get('tags'):
                print(f"   Tags: {', '.join(article['tags'])}")
            print()

# Example usage
if __name__ == "__main__":
    scraper = KDNuggetsScraper()
    
    # Basic scraping (faster)
    articles = scraper.scrape_news_page(max_articles=10)
    scraper.print_summary(articles)
    scraper.save_to_json(articles)
    df = scraper.to_dataframe(articles)
    print(df.head())

Scraping https://www.kdnuggets.com/news/index.html...
Found 10 articles

=== SCRAPED 10 ARTICLES ===

1. A Beginner’s Guide to Mastering Gemini + Google Sheets
   URL: https://www.kdnuggets.com/a-beginners-guide-to-mastering-gemini-google-sheets
   Description: In this article, we'll go through the implementation of Gemini with Google Sheets....
   Author: Cornellius Yudha Wijaya

2. How to Combine Streamlit, Pandas, and Plotly for Interactive Data Apps
   URL: https://www.kdnuggets.com/how-to-combine-streamlit-pandas-and-plotly-for-interactive-data-apps
   Description: With just two Python files and a handful of methods, you can build a complete dashboard that rivals ...
   Author: Vinod Chugani

3. How to Learn AI for Data Analytics in 2025
   URL: https://www.kdnuggets.com/how-to-learn-ai-for-data-analytics-in-2025
   Description: Learn these AI tools to stay relevant as a data professional in 2025....
   Author: Natassha Selvaraj

4. Automate Data Quality Reports with n8n: From CSV

In [11]:
df

,URL,Title,Description,Author,Date,Tags
0,https://www.kdnuggets.com/a-beginners-guide-to...,A Beginner’s Guide to Mastering Gemini + Googl...,"In this article, we'll go through the implemen...",Cornellius Yudha Wijaya,,
1,https://www.kdnuggets.com/how-to-combine-strea...,"How to Combine Streamlit, Pandas, and Plotly f...",With just two Python files and a handful of me...,Vinod Chugani,,
2,https://www.kdnuggets.com/how-to-learn-ai-for-...,How to Learn AI for Data Analytics in 2025,Learn these AI tools to stay relevant as a dat...,Natassha Selvaraj,,
3,https://www.kdnuggets.com/automate-data-qualit...,Automate Data Quality Reports with n8n: From C...,Analyze any CSV dataset from a URL and generat...,Vinod Chugani,,
4,https://www.kdnuggets.com/7-popular-llms-expla...,7 Popular LLMs Explained in 7 Minutes,"Get a quick overview of GPT, BERT, LLaMA, and ...",Kanwal Mehreen,,
5,https://www.kdnuggets.com/vibe-coding-a-speed-...,Vibe Coding a Speed Reading App with Python in...,"Less scrolling, more focus. In this 15-minute ...",Bala Priya C,,
6,https://www.kdnuggets.com/10-free-ai-tools-tha...,10 FREE AI Tools That’ll Save You 10+ Hours a ...,"No tech skills needed. Just tools that work, f...",Kanwal Mehreen,,
7,https://www.kdnuggets.com/make-sense-of-a-10k-...,Make Sense of a 10K+ Line GitHub Repos Without...,No time to read huge GitHub projects? This too...,Kanwal Mehreen,,
8,https://www.kdnuggets.com/build-a-data-cleanin...,Build a Data Cleaning & Validation Pipeline in...,Clean and validate messy data with a compact P...,Bala Priya C,,
9,https://www.kdnuggets.com/building-ai-agent-wi...,Building AI Agents with llama.cpp,This guide will walk you through the entire pr...,Abid Ali Awan,,
